In [ ]:
%pip install -U langchain langchain-google-genai pandas

import os
import pandas as pd
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool

os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6Lta8Tid0Uk9KZAmCW7uwsPt0vJ5ASAVt22Yebhr3hNOw"

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant."),
    ("human", "{question}")
])

chain = prompt | llm

products = pd.DataFrame({
    "product": ["Laptop A", "Laptop B", "Smartphone A", "Smartphone B", "Headphones A"],
    "price": [85000, 120000, 65000, 95000, 15000],
    "currency": ["PKR", "PKR", "PKR", "PKR", "PKR"]
})

products.to_csv("products.csv", index=False)

@tool
def calculator(a: float, b: float, operation: str) -> float:
    """Perform addition, subtraction, multiplication, or division on two numbers."""
    
    operation = operation.strip().lower()

    if operation in ["add", "addition", "plus"]:
        return a + b

    elif operation in ["subtract", "subtraction", "minus"]:
        return a - b

    elif operation in ["multiply", "multiplication", "times"]:
        return a * b

    elif operation in ["divide", "division"]:
        if b == 0:
            raise ValueError("Cannot divide by zero.")
        return a / b

    else:
        raise ValueError(
            f"Unknown operation: {operation}. "
            "Use add, subtract, multiply, or divide."
        )

print("Calculator tool updated successfully.")
@tool
def get_weather(city: str) -> dict:
    """Return weather information including temperature and condition for a supported city."""
    weather_data = {
        "karachi": {"temperature": 34, "condition": "Sunny"},
        "lahore": {"temperature": 31, "condition": "Partly cloudy"},
        "islamabad": {"temperature": 27, "condition": "Cloudy"},
        "dubai": {"temperature": 38, "condition": "Sunny"}
    }
    city_key = city.strip().lower()
    if city_key not in weather_data:
        raise ValueError(f"Weather data not available for {city}.")
    return weather_data[city_key]

@tool
def get_product_price(product_name: str) -> dict:
    """Read a product price from products.csv and return its price and currency."""
    data = pd.read_csv("products.csv")
    match = data[data["product"].str.lower() == product_name.strip().lower()]
    if match.empty:
        raise ValueError(f"Product not found: {product_name}")
    row = match.iloc[0]
    return {
        "product": row["product"],
        "price": float(row["price"]),
        "currency": row["currency"]
    }

tools = [calculator, get_weather, get_product_price]

print("Task 1 + Task 2 setup completed successfully.")
print("LLM:", type(llm).__name__)
print("Tools:", [tool.name for tool in tools])

print("\nCalculator Test:")
print(calculator.invoke({"a": 20, "b": 5, "operation": "multiply"}))

print("\nWeather Test:")
print(get_weather.invoke({"city": "Karachi"}))

print("\nProduct Price Test:")
print(get_product_price.invoke({"product_name": "Laptop A"}))

Note: you may need to restart the kernel to use updated packages.
Task 1 + Task 2 setup completed successfully.
LLM: ChatGoogleGenerativeAI
Tools: ['calculator', 'get_weather', 'get_product_price']

Calculator Test:
100.0

Weather Test:
{'temperature': 34, 'condition': 'Sunny'}

Product Price Test:
{'product': 'Laptop A', 'price': 85000.0, 'currency': 'PKR'}


# Task 3: Build Agent with Tools

In this task, the Gemini language model from Task 1 will be connected with the tools created in Task 2. A LangChain tool-calling agent and AgentExecutor will be used to allow the model to select and execute tools automatically.
The agent will be tested with a multi-step request so that the complete tool execution process can be observed.

In [20]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""You are a helpful AI assistant with access to tools.
Use the available tools whenever needed.
Use calculator for mathematical calculations.
Use get_weather for weather information.
Use get_product_price for product prices.
Always provide a clear final answer based on the tool results."""
)

print("LangChain agent created successfully.")

LangChain agent created successfully.


In [ ]:
# Test Simple Tool Call
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the weather in Karachi?"
        }
    ]
})

print("Agent execution completed.")
print("\nFinal response:")
print(result["messages"][-1].content)

Agent execution completed.

Final response:
The weather in Karachi is Sunny with a temperature of 34 degrees Celsius.


In [25]:
from langchain_core.tools import tool

@tool
def calculator(a: float, b: float, operation: str) -> float:
    """Perform addition, subtraction, multiplication, or division on two numbers."""

    operation = operation.strip().lower()

    if operation in ["add", "addition", "plus"]:
        return a + b

    elif operation in ["subtract", "subtraction", "minus"]:
        return a - b

    elif operation in ["multiply", "multiplication", "times"]:
        return a * b

    elif operation in ["divide", "division"]:
        if b == 0:
            raise ValueError("Cannot divide by zero.")
        return a / b

    else:
        raise ValueError(
            f"Unknown operation '{operation}'. "
            "Use add, subtract, multiply, or divide."
        )

print("New calculator loaded successfully.")
print(calculator.invoke({
    "a": 85000,
    "b": 10,
    "operation": "multiply"
}))

New calculator loaded successfully.
850000.0


In [27]:
from langchain.agents import create_agent

tools = [
    calculator,
    get_weather,
    get_product_price
]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""You are a helpful AI assistant with access to tools.

Use the available tools whenever needed.

Use get_product_price for product prices.
Use calculator for mathematical calculations.
Use get_weather for weather information.

For calculator operations, use exactly:
add, subtract, multiply, or divide.

Always provide a clear final answer based on the tool results."""
)

print("Agent recreated successfully.")
print("Registered tools:", [t.name for t in tools])

Agent recreated successfully.
Registered tools: ['calculator', 'get_weather', 'get_product_price']


In [28]:
multi_step_result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": """Find the price of Laptop A.
Then calculate its price after adding 10%.
Finally, tell me the weather in Karachi."""
        }
    ]
})

print("Multi-step agent execution completed.")
print("\nFinal Answer:")
print(multi_step_result["messages"][-1].content)

Multi-step agent execution completed.

Final Answer:
[{'type': 'text', 'text': 'The price of Laptop A is 85000 PKR. After adding 10%, its price will be 93500 PKR. The weather in Karachi is Sunny with a temperature of 34 degrees Celsius.', 'extras': {'signature': 'Co4CARFNMg8NY5LaHSg6x/Z3S4wEmW9kTnvOfzyKiNiDZajXxuNumTKkaW7lJgr6daT9TxrJ2lBQ7lzBaF27gdByTXAize8MGttn62xCdiebqKXxXuvA74nBoyIfDH2YJIrp3E1WDmZbHD20ZBADmb8Avt7VuL1TyIf0D74ajdgrLHIGqTUIinzrDtd4+WkCOeBEibX8BN/7bJrHsNYkrRKQQftwCIV/LTipz/kqB4BPm9R2Igyqpeekf6oRK571Ba0VRyOjyxu1lwShGuAJ/5VYv+H40GtatBq25FR49GFIOw+omS2csGgZjVNYE1fwZ71RBg7yDsQ9b+KlB+HV0BluesXkR6UaxPxw95hoH5QK'}}]


In [29]:
print("COMPLETE AGENT EXECUTION TRACE\n")

for i, message in enumerate(multi_step_result["messages"], 1):
    print(f"\n--- Message {i} ---")
    print("Type:", type(message).__name__)
    
    if hasattr(message, "content"):
        print("Content:", message.content)
    
    if hasattr(message, "tool_calls") and message.tool_calls:
        print("Tool Calls:")
        for call in message.tool_calls:
            print(call)

COMPLETE AGENT EXECUTION TRACE


--- Message 1 ---
Type: HumanMessage
Content: Find the price of Laptop A.
Then calculate its price after adding 10%.
Finally, tell me the weather in Karachi.

--- Message 2 ---
Type: AIMessage
Content: 
Tool Calls:
{'name': 'get_product_price', 'args': {'product_name': 'Laptop A'}, 'id': '7a431aac-4dc0-4eb2-a060-17a243677eae', 'type': 'tool_call'}

--- Message 3 ---
Type: ToolMessage
Content: {"product": "Laptop A", "price": 85000.0, "currency": "PKR"}

--- Message 4 ---
Type: AIMessage
Content: 
Tool Calls:
{'name': 'calculator', 'args': {'b': 0.1, 'operation': 'multiply', 'a': 85000}, 'id': '26ab7608-b69e-4cfa-8475-f7784632c02a', 'type': 'tool_call'}

--- Message 5 ---
Type: ToolMessage
Content: 8500.0

--- Message 6 ---
Type: AIMessage
Content: 
Tool Calls:
{'name': 'calculator', 'args': {'operation': 'add', 'b': 8500, 'a': 85000}, 'id': 'bb213d11-1bd6-4137-a86e-a913843ffa35', 'type': 'tool_call'}

--- Message 7 ---
Type: ToolMessage
Content: 93500.0

In [30]:
print("ANNOTATED AGENT TRACE\n")

for i, message in enumerate(multi_step_result["messages"], 1):
    print(f"\n--- Step {i} ---")
    print("Message Type:", type(message).__name__)

    if hasattr(message, "tool_calls") and message.tool_calls:
        print("REASON: The agent decided that a tool was required.")
        print("ACT: The agent called the following tool(s):")
        for call in message.tool_calls:
            print("  Tool:", call["name"])
            print("  Arguments:", call["args"])

    elif type(message).__name__ == "ToolMessage":
        print("OBSERVE: The agent received the tool result.")
        print("Result:", message.content)

    else:
        print("FINAL/AGENT MESSAGE:")
        print(message.content)

ANNOTATED AGENT TRACE


--- Step 1 ---
Message Type: HumanMessage
FINAL/AGENT MESSAGE:
Find the price of Laptop A.
Then calculate its price after adding 10%.
Finally, tell me the weather in Karachi.

--- Step 2 ---
Message Type: AIMessage
REASON: The agent decided that a tool was required.
ACT: The agent called the following tool(s):
  Tool: get_product_price
  Arguments: {'product_name': 'Laptop A'}

--- Step 3 ---
Message Type: ToolMessage
OBSERVE: The agent received the tool result.
Result: {"product": "Laptop A", "price": 85000.0, "currency": "PKR"}

--- Step 4 ---
Message Type: AIMessage
REASON: The agent decided that a tool was required.
ACT: The agent called the following tool(s):
  Tool: calculator
  Arguments: {'b': 0.1, 'operation': 'multiply', 'a': 85000}

--- Step 5 ---
Message Type: ToolMessage
OBSERVE: The agent received the tool result.
Result: 8500.0

--- Step 6 ---
Message Type: AIMessage
REASON: The agent decided that a tool was required.
ACT: The agent called the fol

In [38]:
# Tool Usage Summary
from collections import Counter

tool_usage = []

for message in multi_step_result["messages"]:
    if hasattr(message, "tool_calls") and message.tool_calls:
        for call in message.tool_calls:
            tool_usage.append(call["name"])

print("TOOL USAGE SUMMARY")
print("-" * 30)

counts = Counter(tool_usage)

for tool_name, count in counts.items():
    print(f"{tool_name}: {count} call(s)")

print("\nTotal tool calls:", len(tool_usage))

TOOL USAGE SUMMARY
------------------------------
get_product_price: 1 call(s)
calculator: 2 call(s)
get_weather: 1 call(s)

Total tool calls: 4


In [43]:
print("DETAILED TOOL CALL ANALYSIS")

step_number = 1

for message in multi_step_result["messages"]:
    if hasattr(message, "tool_calls") and message.tool_calls:
        for call in message.tool_calls:
            print(f"\nTool Call {step_number}")
            print("Tool:", call["name"])
            print("Arguments:", call["args"])
            print("Tool Call ID:", call["id"])
            step_number += 1

DETAILED TOOL CALL ANALYSIS

Tool Call 1
Tool: get_product_price
Arguments: {'product_name': 'Laptop A'}
Tool Call ID: 7a431aac-4dc0-4eb2-a060-17a243677eae

Tool Call 2
Tool: calculator
Arguments: {'b': 0.1, 'operation': 'multiply', 'a': 85000}
Tool Call ID: 26ab7608-b69e-4cfa-8475-f7784632c02a

Tool Call 3
Tool: calculator
Arguments: {'operation': 'add', 'b': 8500, 'a': 85000}
Tool Call ID: bb213d11-1bd6-4137-a86e-a913843ffa35

Tool Call 4
Tool: get_weather
Arguments: {'city': 'Karachi'}
Tool Call ID: 1fd03386-132e-48b1-bd51-2a0c6ae66c08


In [44]:
print("TOOL OBSERVATIONS")

for i, message in enumerate(multi_step_result["messages"], 1):
    if type(message).__name__ == "ToolMessage":
        print(f"\nObservation {i}")
        print("Tool:", message.name)
        print("Result:", message.content)
        print("Status:", getattr(message, "status", "success"))

TOOL OBSERVATIONS

Observation 3
Tool: get_product_price
Result: {"product": "Laptop A", "price": 85000.0, "currency": "PKR"}
Status: success

Observation 5
Tool: calculator
Result: 8500.0
Status: success

Observation 7
Tool: calculator
Result: 93500.0
Status: success

Observation 9
Tool: get_weather
Result: {"temperature": 34, "condition": "Sunny"}
Status: success


In [45]:
print("AGENT STATE SUMMARY")

messages = multi_step_result["messages"]

print("Total messages:", len(messages))

message_types = Counter(type(message).__name__ for message in messages)

print("\nMessage types:")
for message_type, count in message_types.items():
    print(f"{message_type}: {count}")

print("\nFinal message:")
print(messages[-1].content)

AGENT STATE SUMMARY
Total messages: 10

Message types:
HumanMessage: 1
AIMessage: 5
ToolMessage: 4

Final message:
[{'type': 'text', 'text': 'The price of Laptop A is 85000 PKR. After adding 10%, its price will be 93500 PKR. The weather in Karachi is Sunny with a temperature of 34 degrees Celsius.', 'extras': {'signature': 'Co4CARFNMg8NY5LaHSg6x/Z3S4wEmW9kTnvOfzyKiNiDZajXxuNumTKkaW7lJgr6daT9TxrJ2lBQ7lzBaF27gdByTXAize8MGttn62xCdiebqKXxXuvA74nBoyIfDH2YJIrp3E1WDmZbHD20ZBADmb8Avt7VuL1TyIf0D74ajdgrLHIGqTUIinzrDtd4+WkCOeBEibX8BN/7bJrHsNYkrRKQQftwCIV/LTipz/kqB4BPm9R2Igyqpeekf6oRK571Ba0VRyOjyxu1lwShGuAJ/5VYv+H40GtatBq25FR49GFIOw+omS2csGgZjVNYE1fwZ71RBg7yDsQ9b+KlB+HV0BluesXkR6UaxPxw95hoH5QK'}}]


## Task 3 Conclusion

The LangChain agent was successfully built using Google Gemini and three registered tools: calculator, weather, and product price lookup. The agent completed multi-step tasks by selecting the appropriate tools, executing them, observing their results, and generating a final response. Compared with the raw Python agent from Day 5, LangChain reduced the amount of manual agent-loop and tool-orchestration code required. The execution trace also demonstrated the Reason → Act → Observe workflow, while showing that LangChain manages much of this process internally.
